# Tensors: PyTorch's core data structure

_PyTorch (a complete course)_

**A tensor is an n-dimensional array, like a numpy array but with GPU and autograd built in.**

A tensor is just a flat block of numbers in memory plus a small amount of bookkeeping: its shape (how many along each dimension), its dtype (what kind of number — float32, int64, bool), and its device (where the numbers live — cpu or cuda). Everything else is operations on that block.

---

This notebook is a practice scaffold for the **Tensors: PyTorch's core data structure** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import numpy as np

torch.manual_seed(0)  # reproducible randn

# ---- CREATION: many ways to make a tensor ----
a = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])   # from a Python list -> float32 by default
z = torch.zeros(2, 3)                 # all zeros
o = torch.ones(2, 3)                  # all ones
r = torch.arange(0, 12)               # 0,1,2,...,11  (a range, int64)
g = torch.randn(2, 3)                 # standard-normal random
I = torch.eye(3)                      # 3x3 identity matrix

print("a =\n", a)
print("arange:", r)
print("identity:\n", I)

# ---- SHAPE / DTYPE / DEVICE: the bookkeeping on every tensor ----
print("shape :", a.shape)             # torch.Size([2, 3])
print("ndim  :", a.ndim)              # 2
print("numel :", a.numel())           # 6 total elements
print("dtype :", a.dtype)             # torch.float32  (the default for floats)
print("device:", a.device)            # cpu

# ---- DTYPES: float32 default, and converting ----
i = torch.tensor([1, 2, 3])           # int literals -> int64
print("int tensor dtype :", i.dtype)  # torch.int64
f = i.to(torch.float32)               # cast to float32
print("after .to(float32):", f.dtype) # torch.float32
b = torch.tensor([1, 0, 2]).bool()    # nonzero -> True
print("bool tensor:", b)

# ---- NUMPY ROUND-TRIP + the SHARED-MEMORY gotcha ----
npy = np.array([1.0, 2.0, 3.0])       # numpy defaults to float64
t = torch.from_numpy(npy)             # SHARES the same memory buffer
print("from_numpy dtype:", t.dtype)   # torch.float64 (note: not float32!)
t[0] = 99.0                           # mutate the torch tensor...
print("numpy array is now:", npy)     # ...and the numpy array changed too: [99. 2. 3.]
safe = torch.tensor(npy)              # torch.tensor COPIES -> independent
back = a.numpy()                      # .numpy() also shares memory (CPU tensors)
print("back to numpy shape:", back.shape)

# ---- DEVICES: move to GPU if one is available ----
device = "cuda" if torch.cuda.is_available() else "cpu"
print("using device:", device)
a_dev = a.to(device)                  # copy onto the chosen device
print("a is on:", a_dev.device)
a_back = a_dev.cpu()                  # bring it back to CPU (e.g. before .numpy())

# ---- INDEXING / SLICING: numpy-like ----
print("first row      :", a[0])       # [1., 2., 3.]
print("second column  :", a[:, 1])    # [2., 5.]
print("top-left 2x2    :\n", a[:2, :2])
print("positive mask  :", a[a > 3])   # boolean indexing -> [4., 5., 6.]
print("single element :", a[0, 2].item())  # .item() -> plain Python float 3.0


## Visualize the data & results

_How much memory does the same 1000-element tensor take in each dtype?_

In [ ]:
import numpy as np

n = 1000  # number of elements in the tensor
dtypes = ["float64", "float32", "float16", "int8"]
bytes_per = {"float64": 8, "float32": 4, "float16": 2, "int8": 1}

# total bytes = elements * bytes-per-element (this is exactly what tensor.element_size()
# times tensor.numel() reports in PyTorch)
totals = [n * bytes_per[d] for d in dtypes]
for d, t in zip(dtypes, totals):
    print(f"{d:8s}: {t} bytes")
# float64 : 8000 bytes
# float32 : 4000 bytes
# float16 : 2000 bytes
# int8    : 1000 bytes

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You have a numpy array a = np.array([1, 2, 3]) and run t = torch.from_numpy(a). Then you do t[0] = 99. What is a[0] afterward, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what from_numpy does to memory. — _It wraps the same buffer; it does not copy._
- Mutate t, then read a. — _Both names point at one block of numbers._

**Answer:** a[0] is now 99. from_numpy shares memory, so writing through the tensor also changes the numpy array. Use torch.tensor(a) or t.clone() if you need an independent copy.

</details>

**Problem 2.** You write w = torch.tensor([1, 2, 3], requires_grad=True) and PyTorch raises an error. What is wrong and how do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check the dtype of [1, 2, 3]. — _Integer literals give an int64 tensor._
- Recall which tensors can carry gradients. — _Only floating-point tensors can require grad._

**Answer:** Integers cannot have gradients. Make it a float: torch.tensor([1.0, 2.0, 3.0], requires_grad=True) (or add dtype=torch.float32).

</details>

**Problem 3.** A loss tensor prints as tensor(0.4231). You append it to a Python list every step to plot later, and memory grows each epoch. What is the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice the tensor still carries its autograd graph. — _Keeping the tensor keeps the whole computation graph alive._
- Extract the plain number instead. — _.item() returns a detached Python float._

**Answer:** Store loss.item(), not loss. .item() pulls out a plain Python float and drops the reference to the graph, so old graphs can be freed.

</details>